In [23]:
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [34]:
df = pd.read_csv("adult.csv")


df = df.replace("?", np.nan).dropna().reset_index(drop=True)

X = df.drop(columns=["income"])
y = df["income"].map({"<=50K": 0, ">50K": 1})

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Dataset shape after cleaning:", df.shape)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Target balance:")
print(y.value_counts(normalize=True))


Dataset shape after cleaning: (45222, 15)
Training rows: 36177
Testing rows: 9045
Target balance:
income
0    0.752156
1    0.247844
Name: proportion, dtype: float64


In [25]:
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", one_hot, categorical_features)
    ],
    sparse_threshold=0
)


In [26]:
def make_pipeline(model):
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

baseline_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": LinearSVC(max_iter=5000, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}


def evaluate_models(models, X_train, X_test, y_train, y_test):
    rows = []
    fitted_models = {}

    for name, model in models.items():
        pipe = make_pipeline(model)

        start = time.time()
        pipe.fit(X_train, y_train)
        training_time = time.time() - start

        y_pred = pipe.predict(X_test)

        rows.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred),
            "Training Time (s)": training_time
        })
        fitted_models[name] = pipe

    results = pd.DataFrame(rows).sort_values(
        by=["F1 Score", "Accuracy"],
        ascending=False
    ).reset_index(drop=True)

    return results, fitted_models


In [27]:
before_tuning_results, baseline_fitted_models = evaluate_models(
    baseline_models,
    X_train,
    X_test,
    y_train,
    y_test
)

before_tuning_results


,Model,Accuracy,F1 Score,Training Time (s)
0,XGBoost,0.863460,0.694383,0.210778
1,Random Forest,0.844555,0.659894,0.563460
2,Logistic Regression,0.845992,0.658160,0.364622
3,SVM,0.845661,0.654284,0.196852
4,KNN,0.824876,0.625355,0.051991
5,Decision Tree,0.805970,0.617230,0.358527


In [28]:
param_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1, 10]
    },
    "Decision Tree": {
        "model__max_depth": [None, 5, 10, 20],
        "model__min_samples_split": [2, 10]
    },
    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20]
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 11]
    },
    "SVM": {
        "model__C": [0.1, 1, 10]
    },
    "XGBoost": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [3, 5],
        "model__learning_rate": [0.05, 0.1]
    }
}

best_models = {}
tuning_rows = []

for name, model in baseline_models.items():
    print(f"Tuning {name}...")
    grid = GridSearchCV(
        estimator=make_pipeline(model),
        param_grid=param_grids[name],
        scoring="f1",
        cv=3,
        n_jobs=-1
    )

    start = time.time()
    grid.fit(X_train, y_train)
    tuning_time = time.time() - start

    best_models[name] = grid.best_estimator_.named_steps["model"]
    tuning_rows.append({
        "Model": name,
        "Best Params": grid.best_params_,
        "Best CV F1": grid.best_score_,
        "Tuning Time (s)": tuning_time
    })

best_params_df = pd.DataFrame(tuning_rows)
best_params_df


Tuning Logistic Regression...
Tuning Decision Tree...
Tuning Random Forest...
Tuning KNN...
Tuning SVM...
Tuning XGBoost...


,Model,Best Params,Best CV F1,Tuning Time (s)
0,Logistic Regression,{'model__C': 1},0.666828,1.952590
1,Decision Tree,"{'model__max_depth': 10, 'model__min_samples_s...",0.667348,2.074649
2,Random Forest,"{'model__max_depth': 20, 'model__n_estimators'...",0.681576,10.028723
3,KNN,{'model__n_neighbors': 11},0.650216,4.853361
4,SVM,{'model__C': 0.1},0.662256,0.966780
5,XGBoost,"{'model__learning_rate': 0.1, 'model__max_dept...",0.715726,3.251413


In [29]:
after_tuning_results, tuned_fitted_models = evaluate_models(
    best_models,
    X_train,
    X_test,
    y_train,
    y_test
)

after_tuning_results


,Model,Accuracy,F1 Score,Training Time (s)
0,XGBoost,0.866224,0.708293,0.503346
1,Random Forest,0.860033,0.680948,0.524564
2,Decision Tree,0.848977,0.660705,0.264124
3,Logistic Regression,0.845992,0.658160,0.413678
4,SVM,0.845771,0.654275,0.209754
5,KNN,0.833831,0.642738,0.055988


In [30]:
comparison = before_tuning_results.merge(
    after_tuning_results,
    on="Model",
    suffixes=(" Before", " After")
)

comparison["Accuracy Change"] = comparison["Accuracy After"] - comparison["Accuracy Before"]
comparison["F1 Change"] = comparison["F1 Score After"] - comparison["F1 Score Before"]
comparison["Training Time Change (s)"] = comparison["Training Time (s) After"] - comparison["Training Time (s) Before"]

comparison = comparison.sort_values(
    by=["F1 Score After", "Accuracy After"],
    ascending=False
).reset_index(drop=True)

comparison


,Model,Accuracy Before,F1 Score Before,Training Time (s) Before,Accuracy After,F1 Score After,Training Time (s) After,Accuracy Change,F1 Change,Training Time Change (s)
0,XGBoost,0.863460,0.694383,0.210778,0.866224,0.708293,0.503346,0.002764,0.013911,0.292569
1,Random Forest,0.844555,0.659894,0.563460,0.860033,0.680948,0.524564,0.015478,0.021054,-0.038896
2,Decision Tree,0.805970,0.617230,0.358527,0.848977,0.660705,0.264124,0.043007,0.043475,-0.094404
3,Logistic Regression,0.845992,0.658160,0.364622,0.845992,0.658160,0.413678,0.000000,0.000000,0.049055
4,SVM,0.845661,0.654284,0.196852,0.845771,0.654275,0.209754,0.000111,-0.000009,0.012902
5,KNN,0.824876,0.625355,0.051991,0.833831,0.642738,0.055988,0.008955,0.017384,0.003997


In [31]:
leaderboard = after_tuning_results[["Model", "Accuracy", "F1 Score", "Training Time (s)"]].copy()
leaderboard = leaderboard.sort_values(
    by=["F1 Score", "Accuracy", "Training Time (s)"],
    ascending=[False, False, True]
).reset_index(drop=True)
leaderboard.index = leaderboard.index + 1

leaderboard


,Model,Accuracy,F1 Score,Training Time (s)
1,XGBoost,0.866224,0.708293,0.503346
2,Random Forest,0.860033,0.680948,0.524564
3,Decision Tree,0.848977,0.660705,0.264124
4,Logistic Regression,0.845992,0.658160,0.413678
5,SVM,0.845771,0.654275,0.209754
6,KNN,0.833831,0.642738,0.055988


In [32]:
winner = leaderboard.iloc[0]

print("Winner:", winner["Model"])
print(f"Accuracy: {winner['Accuracy']:.4f}")
print(f"F1 Score: {winner['F1 Score']:.4f}")
print(f"Training Time: {winner['Training Time (s)']:.4f} seconds")
print()
print(
    f"{winner['Model']} is selected as the winner because it has the best F1-score "
    f"on the same test dataset. F1-score is a strong choice here because the income "
    f"classes are imbalanced, so it balances precision and recall better than accuracy alone. "
    f"Its accuracy and training time are also included in the leaderboard to confirm the model "
    f"is both effective and practical."
)


Winner: XGBoost
Accuracy: 0.8662
F1 Score: 0.7083
Training Time: 0.5033 seconds

XGBoost is selected as the winner because it has the best F1-score on the same test dataset. F1-score is a strong choice here because the income classes are imbalanced, so it balances precision and recall better than accuracy alone. Its accuracy and training time are also included in the leaderboard to confirm the model is both effective and practical.


In [33]:
before_tuning_results.to_csv("ml_14_before_tuning_results.csv", index=False)
after_tuning_results.to_csv("ml_14_after_tuning_results.csv", index=False)
comparison.to_csv("ml_14_before_after_comparison.csv", index=False)
leaderboard.to_csv("ml_14_final_leaderboard.csv", index=True)

print("Saved result CSV files for Part 1 and Part 2.")


Saved result CSV files for Part 1 and Part 2.
